Рекуррентная нейронная сеть. Генератор текста.

In [ ]:
# Adapted from lstm_text_generation.py in keras/examples
from keras.layers import SimpleRNN
from keras.models import Sequential
from keras.layers import Dense, Activation
import numpy as np

Загружаем файл с текстом.

strip -- удаляет пробелы в начале и конце строки

d.join(s) -- преобразует список s со строками в строку, между строками ставится разделитель d

In [ ]:

INPUT_FILE = "ins4_text.txt"

# extract the input as a stream of characters
print("Extracting text from input...")
fin = open(INPUT_FILE, 'rb')
lines = []
for line in fin:
    line = line.strip().lower()
    line = line.decode("utf-8", "ignore")
    if len(line) == 0:
        continue
    lines.append(line)
fin.close()
text = " ".join(lines)

Extracting text from input...


Создаем прямую и обратную кодовые талицы для всех символов текста

enumerate(chars) -- для каждого x из chars генерирует пару (номер, x), т.е. нумерует все элементы chars

In [ ]:
# creating lookup tables
# Here chars is the number of features in our character "vocabulary"
chars = set([c for c in text])
nb_chars = len(chars)
char2index = dict((c, i) for i, c in enumerate(chars))
index2char = dict((i, c) for i, c in enumerate(chars))

Формируем список строк input_chars и список символов label_chars, стоящих следом за соответствующей строкой. Логика обучения дальше будет такая. Если встретилась строка input_chars[i], то следующим символом надо генерировать label_chars[i]. Так по одному символу генерируем текст.

Пока просто создаем обучающий набор из признаков input_chars и меток label_chars.

In [ ]:
# create inputs and labels from the text. We do this by stepping
# through the text ${step} character at a time, and extracting a
# sequence of size ${seqlen} and the next output char. For example,
# assuming an input text "The sky was falling", we would get the
# following sequence of input_chars and label_chars (first 5 only)
#   The sky wa -> s
#   he sky was ->
#   e sky was  -> f
#    sky was f -> a
#   sky was fa -> l
print("Creating input and label text...")
SEQLEN = 10
STEP = 1

input_chars = []
label_chars = []
for i in range(0, len(text) - SEQLEN, STEP):
    input_chars.append(text[i:i + SEQLEN])
    label_chars.append(text[i + SEQLEN])

Creating input and label text...


Применяем к обучающему набору one-hot кодирование. В данном примере довольно много строк (зависит от загружаемого текста, обозначим это число через N) в обучающем наборе, в каждой строке SEQLEN символов, а всего разных символов nb_chars. Поэтому в результате one-hot кодирования получаем трехмерную матрицу N x SEQLEN x nb_chars из 0 и 1, если точнее из false и true.

In [ ]:
# vectorize the input and label chars
# Each row of the input is represented by seqlen characters, each
# represented as a 1-hot encoding of size len(char). There are
# len(input_chars) such rows, so shape(X) is (len(input_chars),
# seqlen, nb_chars).
# Each row of output is a single character, also represented as a
# dense encoding of size len(char). Hence shape(y) is (len(input_chars),
# nb_chars).
print("Vectorizing input and label text...")
X = np.zeros((len(input_chars), SEQLEN, nb_chars), dtype=np.bool_)
y = np.zeros((len(input_chars), nb_chars), dtype=np.bool_)

for i, input_char in enumerate(input_chars):
    for j, ch in enumerate(input_char):
        X[i, j, char2index[ch]] = 1
    y[i, char2index[label_chars[i]]] = 1

Vectorizing input and label text...


Строим модель рекуррентной нейронной сети.

Первый слой -- SimpleRNN, в котором выходной вектор сигналов подается на вход. HIDDEN_SIZE -- число нейронов на слое. Параметр input_shape используется если это первый слой (наш случай) и задает размерность входного вектора (SEQLEN) и величину временной задержки (timesteps) равной nb_chars, т.е. если обучающий набор состоит из обучающих последовательностей (как раз наш случай, вспомните структуру обучающеего набора), то обучение происходит порциями. Каждая порция равна величине timesteps (в нашем случае это nb_chars).

Второй слой -- персептрон с числом нейронов nb_chars. Т. е. на выходе будет одна 1 и остальные 0. Поскольку каждый нейрон представляет собой один из символов нашего алфавита, то 1 покажет какой именно символ будет сгенерирован в текст.

In [ ]:
# Build the model. We use a single RNN with a fully connected layer
# to compute the most likely predicted output char
HIDDEN_SIZE = 128
BATCH_SIZE = 128
NUM_ITERATIONS = 25
NUM_EPOCHS_PER_ITERATION = 1
NUM_PREDS_PER_EPOCH = 100

model = Sequential()
model.add(SimpleRNN(HIDDEN_SIZE, return_sequences=False,
                    input_shape=(SEQLEN, nb_chars),
                    unroll=True))
model.add(Dense(nb_chars))
model.add(Activation("softmax"))

model.compile(loss="categorical_crossentropy", optimizer="rmsprop")


c:\MyFiles\PyCode\myvenv\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Собственно обучение модели и тестирование на случайной выборке

In [ ]:
# We train the model in batches and test output generated at each step
for iteration in range(NUM_ITERATIONS):
    print("=" * 50)
    print("Iteration #: %d" % (iteration))
    model.fit(X, y, batch_size=BATCH_SIZE, epochs=NUM_EPOCHS_PER_ITERATION)

    # testing model
    # randomly choose a row from input_chars, then use it to
    # generate text from model for next 100 chars
    test_idx = np.random.randint(len(input_chars))
    test_chars = input_chars[test_idx]
    print("Generating from seed: %s" % (test_chars))
    print(test_chars, end="")
    for i in range(NUM_PREDS_PER_EPOCH):
        Xtest = np.zeros((1, SEQLEN, nb_chars))
        for i, ch in enumerate(test_chars):
            Xtest[0, i, char2index[ch]] = 1
        pred = model.predict(Xtest, verbose=0)[0]
        ypred = index2char[np.argmax(pred)]
        print(ypred, end="")
        # move forward with test_chars + ypred
        test_chars = test_chars[1:] + ypred
    print()


Iteration #: 0
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 3.4163  
Generating from seed: е том и дн
е том и днт  уаааУ  нл кг  чздкт тгнтллзззубре рвлийргхалпщанаыйнахйыпщань зг тц  м куит  алакх ухл зрг з уггт
Iteration #: 1
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.0771 
Generating from seed: песнь заво
песнь завод п залувр  рл зу уа   улл  уант  ааац  аана   ааа   уаа   уаа   уаа   уаа   уаа   уаа   уаа   уаа  
Iteration #: 2
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.9452 
Generating from seed: идит там ц
идит там цу е                                                                                                 
Iteration #: 3
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.8146 
Generating from seed: еный все х
еный все х                                                                                                    
Iteration #: 4
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.7129 
Generating from seed: дух там ру
дух там ру  е   е     е   е     е   е     е   е   

Тестирование модели на начальной строке. Интересно сколько символов модель сможет восстановить (запускаем код ниже). Ни одного. Модель плохая? Нет, проверено, модель хорошая. Надо просто подобрать параметры обучения.

In [ ]:
test_chars = "У лукоморь"
print(test_chars, end="")
for i in range(NUM_PREDS_PER_EPOCH):
  Xtest = np.zeros((1, SEQLEN, nb_chars))
  for i, ch in enumerate(test_chars):
    Xtest[0, i, char2index[ch]] = 1
  pred = model.predict(Xtest, verbose=0)[0]
  ypred = index2char[np.argmax(pred)]
  print(ypred, end="")
  # move forward with test_chars + ypred
  test_chars = test_chars[1:] + ypred

У лукоморьз  т   т на   а     е   а     е   а     е   а     е   а     е   а     е   а     е   а     е   а     

1. Исследуйте влияние параметров рекуррентной нейронной сети на результат обучения. Метрикой качества обучения будет количество восстановленных символов, продолжающих начальную строку.

### Изучение гиперпараметров
- **HIDDEN_SIZE** = 128
- **BATCH_SIZE** = 128
- **NUM_ITERATIONS** = 25
- **NUM_EPOCHS_PER_ITERATION** = 1
- **NUM_PREDS_PER_EPOCH** = 100
1. Эти гиперпараметры выступали, как начальные. Попробуем их менять и изучать поведение модели

1. Добавим нейронов
2. Увеличим кол-во эпох обучения
3. Увеличим количестов этапов обучения на эпохе

In [ ]:
SEQLEN = 10
STEP = 1

HIDDEN_SIZE = 256
BATCH_SIZE = 64
NUM_ITERATIONS = 25
NUM_EPOCHS_PER_ITERATION = 100
NUM_PREDS_PER_EPOCH = 100

In [ ]:
model = Sequential()
model.add(SimpleRNN(HIDDEN_SIZE, return_sequences=False,
                    input_shape=(SEQLEN, nb_chars),
                    unroll=True))
model.add(Dense(nb_chars))
model.add(Activation("softmax"))

model.compile(loss="categorical_crossentropy", optimizer="rmsprop")

In [ ]:
# We train the model in batches and test output generated at each step
for iteration in range(NUM_ITERATIONS):
    print("=" * 50)
    print("Iteration #: %d" % (iteration))
    model.fit(X, y, batch_size=BATCH_SIZE, epochs=NUM_EPOCHS_PER_ITERATION)

    # testing model
    # randomly choose a row from input_chars, then use it to
    # generate text from model for next 100 chars
    test_idx = np.random.randint(len(input_chars))
    test_chars = input_chars[test_idx]
    print("Generating from seed: %s" % (test_chars))
    print(test_chars, end="")
    for i in range(NUM_PREDS_PER_EPOCH):
        Xtest = np.zeros((1, SEQLEN, nb_chars))
        for i, ch in enumerate(test_chars):
            Xtest[0, i, char2index[ch]] = 1
        pred = model.predict(Xtest, verbose=0)[0]
        ypred = index2char[np.argmax(pred)]
        print(ypred, end="")
        # move forward with test_chars + ypred
        test_chars = test_chars[1:] + ypred
    print()


Iteration #: 0
Epoch 1/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.3326  
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.8175 
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.5890 
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.3714 
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.1235 
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.0281 
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.9098 
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.7237 
Epoch 9/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.7153 
Epoch 10/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.5886 
Epoch 11/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.4246 
Epoch 12/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.4574 
Epoch 13/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.4110 
Epoch 14/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.2957 
Epoch 15/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.3174 
Epo

In [ ]:
test_chars = "У лукоморь"
initial_chars = test_chars  # сохраняем начальную строку для сравнения

print(test_chars, end="")
restored_count = 0  # счетчик восстановленных символов

for i in range(NUM_PREDS_PER_EPOCH):
    Xtest = np.zeros((1, SEQLEN, nb_chars))
    for i, ch in enumerate(test_chars):
        Xtest[0, i, char2index[ch]] = 1
    pred = model.predict(Xtest, verbose=0)[0]
    ypred = index2char[np.argmax(pred)]
    print(ypred, end="")

    if initial_chars.startswith(test_chars + ypred):
        restored_count += 1

    # обновляем initial_chars
    initial_chars += ypred

    # move forward with test_chars + ypred
    test_chars = test_chars[1:] + ypred

print("\nВосстановленные символы:", restored_count)


У лукоморья дуб зеленый златая цепь на дубе том и днем и ночью кот ученый все ходит по цепи кругом пойдет на п
Восстановленные символы: 0


In [ ]:
text[SEQLEN], SEQLEN, text

('я',
 10,
 'У лукоморья дуб зеленый златая цепь на дубе том и днем и ночью кот ученый все ходит по цепи кругом пойдет на право песнь заводит налево сказку говорит там чудеса там леший бродит русалка на ветвях сидит там царь кощей над златом чахнет там русский дух там русью пахнет')

In [ ]:
test_chars = "У лукоморь"
initial_chars = test_chars  # сохраняем начальную строку для сравнения

print(test_chars, end="")

# Подсчет правильно предсказанных символов
correct_predictions = 0
for i in range(NUM_PREDS_PER_EPOCH):
    Xtest = np.zeros((1, SEQLEN, nb_chars))
    for j, ch in enumerate(test_chars):
        Xtest[0, j, char2index[ch]] = 1
    pred = model.predict(Xtest, verbose=0)[0]
    ypred = index2char[np.argmax(pred)]
    print(ypred, end="")
    if ypred == text[SEQLEN + i].strip():
        correct_predictions += 1
    # Продолжение строки для тестирования
    test_chars = test_chars[1:] + ypred

# Вывод метрики качества на тестовых данных
accuracy = correct_predictions
print(f"\nВосстановленные символы {accuracy}")

У лукоморья дуб зеленый златая цепь на дубе том и днем и ночью кот ученый все ходит по цепи кругом пойдет на п
Восстановленные символы 79


## Вывод
- Модель работает идеально. Стоило лишь добавить доп. эпох и нейронов.